# LingBot-Map on Colab — reliable outputs

## What this notebook does
1. **Always** builds LingBot **`scene.glb`** (interactive 3D) — same path as official `demo.py`
2. **Then tries** official flythrough **`MP4`** via `batch_demo.py` (needs Kaolin; often fails on free Colab)
3. If MP4 fails, you still have the proper LingBot GLB

## Setup
1. Runtime → **T4 GPU**
2. Short indoor video (20–60s)
3. Run top → bottom


## 0. GPU check


In [ ]:
import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Runtime → Change runtime type → T4 GPU, then restart and rerun.")
print("gpu", torch.cuda.get_device_name(0))


## 1. Install LingBot-Map (reconstruction only)


In [ ]:
import os
from pathlib import Path
import sys

%cd /content
if not Path("/content/lingbot-map").exists():
    !git clone --depth 1 https://github.com/Robbyant/lingbot-map.git
%cd /content/lingbot-map
!{sys.executable} -m pip -q install -e ".[vis]" huggingface_hub
print("cwd", os.getcwd())


## 2. Download LingBot weights


In [ ]:
from huggingface_hub import hf_hub_download
MODEL_PATH = hf_hub_download(
    repo_id="robbyant/lingbot-map",
    filename="lingbot-map.pt",
    local_dir="/content/weights",
)
print("checkpoint:", MODEL_PATH)


## 3. Upload your video


In [ ]:
from google.colab import files
from pathlib import Path

IN = Path("/content/my_video")
IN.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
assert uploaded, "Upload one video"
name = list(uploaded.keys())[0]
VIDEO = IN / name
VIDEO.write_bytes(uploaded[name])
print("VIDEO =", VIDEO, round(VIDEO.stat().st_size/1e6, 2), "MB")


## 4. LingBot reconstruction → `scene.glb` (this should always work)

Same as official internship pipeline: load frames → `inference_streaming` → `predictions_to_glb`.

If you OOM, set `FIRST_K = 24`.


In [ ]:
from tqdm.std import tqdm as std_tqdm
import argparse, sys, time
from pathlib import Path
import torch

sys.path.insert(0, "/content/lingbot-map")
import demo as lingbot_demo
lingbot_demo.tqdm = std_tqdm
from lingbot_map.vis.glb_export import predictions_to_glb

FIRST_K = 48
FPS = 10
CONF_THRES = 50.0
OUT_GLB = Path("/content/scene.glb")

device = torch.device("cuda")
images, _, _ = lingbot_demo.load_images(
    video_path=str(VIDEO), fps=FPS, first_k=FIRST_K, image_size=518, patch_size=14
)
num_frames = int(images.shape[0])
scale_frames = min(8, max(1, num_frames - 1))
print(f"frames={num_frames} scale_frames={scale_frames}")

args = argparse.Namespace(
    mode="streaming",
    model_path=str(MODEL_PATH),
    image_size=518,
    patch_size=14,
    enable_3d_rope=True,
    max_frame_num=1024,
    kv_cache_sliding_window=64,
    num_scale_frames=scale_frames,
    use_sdpa=True,
    camera_num_iterations=4,
    window_size=64,
    overlap_size=16,
    overlap_keyframes=None,
)
model = lingbot_demo.load_model(args, device)
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
if getattr(model, "aggregator", None) is not None:
    model.aggregator = model.aggregator.to(dtype=dtype)

images = images.to(device)
t0 = time.time()
with torch.no_grad(), torch.amp.autocast("cuda", dtype=dtype):
    predictions = model.inference_streaming(
        images,
        num_scale_frames=scale_frames,
        keyframe_interval=1,
        output_device=torch.device("cpu"),
    )
print(f"inference {time.time()-t0:.1f}s")

images_for_post = predictions.get("images", images)
predictions, images_cpu = lingbot_demo.postprocess(predictions, images_for_post)
vis = lingbot_demo.prepare_for_visualization(predictions, images_cpu)
scene = predictions_to_glb(vis, conf_thres=CONF_THRES, show_cam=False, mask_sky=False)
scene.export(str(OUT_GLB))
print(f"WROTE {OUT_GLB} ({OUT_GLB.stat().st_size/1e6:.2f} MB)")
print("Open in https://gltf-viewer.donmccurdy.com/")


## 5. Download LingBot GLB now


In [ ]:
from google.colab import files
from pathlib import Path
assert Path("/content/scene.glb").is_file(), "Run Step 4 first"
files.download("/content/scene.glb")
print("Attach this scene.glb in Streamlit — this is the LingBot 3D output.")


## 6. (Optional / fragile) Official flythrough MP4

This needs **Kaolin + CUDA render extensions**. Free Colab often fails here.
We capture the full error log so you can see why.


In [ ]:
import os, sys, subprocess
from pathlib import Path

!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
%cd /content/lingbot-map
!{sys.executable} -m pip -q install "numpy==1.26.4" open3d==0.19.0 pyyaml onnxruntime-gpu

import torch
ver = torch.__version__.split("+")[0]
kaolin_ok = False
for tag in ["cu128", "cu126", "cu124", "cu121"]:
    url = f"https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-{ver}_{tag}.html"
    print("try Kaolin", url)
    rc = os.system(f'{sys.executable} -m pip -q install --index-url https://pypi.org/simple kaolin -f "{url}"')
    if rc == 0:
        try:
            import kaolin
            print("Kaolin OK")
            kaolin_ok = True
            break
        except Exception as e:
            print("import fail", e)

if not kaolin_ok:
    print("SKIP MP4: Kaolin not available on this Colab. Keep using scene.glb.")
else:
    %cd /content/lingbot-map/demo_render/render_cuda_ext
    ext = subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], capture_output=True, text=True)
    print(ext.stdout[-2000:])
    print(ext.stderr[-2000:])
    %cd /content/lingbot-map

    OUT = Path("/content/lingbot_out")
    OUT.mkdir(parents=True, exist_ok=True)
    CONFIG = "/content/lingbot-map/demo_render/config/indoor.yaml"
    cmd = [
        sys.executable, "demo_render/batch_demo.py",
        "--video_path", str(VIDEO),
        "--output_folder", str(OUT),
        "--model_path", str(MODEL_PATH),
        "--config", CONFIG,
        "--mode", "windowed",
        "--window_size", "64",
        "--keyframe_interval", "4",
        "--overlap_keyframes", "8",
        "--camera_vis", "default",
        "--save_predictions",
    ]
    print("Running:", " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print("exit", proc.returncode)
    print("--- STDOUT (tail) ---")
    print(proc.stdout[-4000:])
    print("--- STDERR (tail) ---")
    print(proc.stderr[-4000:])
    mp4s = sorted(OUT.rglob("*.mp4"))
    print("mp4s", mp4s)
    if mp4s:
        from google.colab import files
        files.download(str(mp4s[0]))
    else:
        print("No MP4. Use scene.glb from Step 5 — that is still proper LingBot reconstruction.")
